# Track A Retrain with OOF Mislabel Filtering

목표는 `trackB_oof_mislabel.ipynb`에서 만든 OOF 기반 mislabel score를 사용해 Track A 재학습 후보를 비교하는 것이다.

- R0: baseline checkpoint 그대로 test inference
- R1: OOF mislabel top100 중 baseline train split에 있는 샘플 제거, duplicate weight 미적용
- R2: OOF mislabel top100 중 baseline train split에 있는 샘플 제거, duplicate weight 적용

검증 포인트는 OOF 파일 1366개 정상 생성, best-holdout-AUC checkpoint 기준 OOF 여부, top50/top100/top150의 baseline train/val 분포, v2 score와 OOF score 비교, top100 grid 저장, val subset별 metric 저장, R0/R1/R2 submission 차이 비교이다.

## 0. Imports and Seed

In [1]:
from pathlib import Path
import json
import random
import time
from datetime import datetime
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights

try:
    from torchinfo import summary
except Exception:
    summary = None

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


device: cuda
gpu: NVIDIA GeForce RTX 4070 SUPER


## 1. Paths and Experiment Config

In [2]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "Quest02":
    PROJECT_DIR = Path("/home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02")

DATA_DIR = PROJECT_DIR / "data" / "RS18A"
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
TRAIN_LABELS_PATH = DATA_DIR / "train_labels.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

BASELINE_CHECKPOINT_PATH = PROJECT_DIR / "model" / "base0_resnet18_finetune_best.pt"
BASELINE_HISTORY_PATH = PROJECT_DIR / "model" / "base0_resnet18_finetune_history.csv"

OOF_DIR = PROJECT_DIR / "outputs" / "trackB_oof_mislabel"
OOF_SCORE_PATH = OOF_DIR / "oof_mislabel_scores.csv"
OOF_CONFIG_PATH = OOF_DIR / "trackB_oof_mislabel_config.json"
OOF_FOLD_ASSIGNMENT_PATH = OOF_DIR / "fold_assignments.csv"
OOF_FOLD_HISTORY_PATH = OOF_DIR / "fold_train_history.csv"
OOF_CHECKPOINT_DIR = OOF_DIR / "checkpoints"

V2_DETAIL_CANDIDATES = [
    PROJECT_DIR / "outputs" / "trackB_070103_v2_20260701_223101" / "trackB_v2_scores_detail.csv",
    PROJECT_DIR / "outputs" / "trackB_070103_v2" / "trackB_v2_scores_detail.csv",
]
V2_DETAIL_PATH = next((path for path in V2_DETAIL_CANDIDATES if path.exists()), None)

RUN_NAME = "trackA_retrain_oof_top100_" + datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = PROJECT_DIR / "outputs" / RUN_NAME
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
TABLE_DIR = OUTPUT_DIR / "tables"
VIS_DIR = OUTPUT_DIR / "visualizations"
SUBMISSION_DIR = OUTPUT_DIR / "submissions"
for path in [OUTPUT_DIR, CHECKPOINT_DIR, TABLE_DIR, VIS_DIR, SUBMISSION_DIR]:
    path.mkdir(parents=True, exist_ok=True)

VAL_RATIO = 0.2
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

TOPK_REMOVE = 100
TOPK_REPORTS = [50, 100, 150]
EXPECTED_OOF_CHECKPOINT = "best_holdout_auc"
REBUILD_BEST_AUC_OOF_IF_NEEDED = True
REQUIRE_BEST_AUC_OOF = True

DUP_WEIGHT_MIN = 0.50
DUP_WEIGHT_POWER = 1.0
RUN_R1 = True
RUN_R2 = True

CONFIG_PATH = OUTPUT_DIR / "trackA_retrain_config.json"
config = {
    "seed": SEED,
    "run_name": RUN_NAME,
    "data_dir": str(DATA_DIR),
    "baseline_checkpoint_path": str(BASELINE_CHECKPOINT_PATH),
    "oof_score_path": str(OOF_SCORE_PATH),
    "oof_config_path": str(OOF_CONFIG_PATH),
    "expected_oof_checkpoint": EXPECTED_OOF_CHECKPOINT,
    "rebuild_best_auc_oof_if_needed": REBUILD_BEST_AUC_OOF_IF_NEEDED,
    "require_best_auc_oof": REQUIRE_BEST_AUC_OOF,
    "topk_remove": TOPK_REMOVE,
    "val_ratio": VAL_RATIO,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "dup_weight_min": DUP_WEIGHT_MIN,
    "dup_weight_power": DUP_WEIGHT_POWER,
    "output_dir": str(OUTPUT_DIR),
}
with CONFIG_PATH.open("w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

required_paths = [
    DATA_DIR,
    TRAIN_DIR,
    TEST_DIR,
    TRAIN_LABELS_PATH,
    SAMPLE_SUBMISSION_PATH,
    BASELINE_CHECKPOINT_PATH,
    OOF_SCORE_PATH,
    OOF_CONFIG_PATH,
    OOF_FOLD_ASSIGNMENT_PATH,
]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("V2_DETAIL_PATH:", V2_DETAIL_PATH)
print("saved config:", CONFIG_PATH)


PROJECT_DIR: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02
OUTPUT_DIR: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_retrain_oof_top100_20260702_093712
V2_DETAIL_PATH: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackB_070103_v2_20260701_223101/trackB_v2_scores_detail.csv
saved config: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_retrain_oof_top100_20260702_093712/trackA_retrain_config.json


## 2. Data, Transform, Model Utilities

In [3]:
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]


def image_path_from_id(image_dir, image_id):
    image_id = str(image_id)
    for ext in IMAGE_EXTENSIONS:
        candidate = Path(image_dir) / f"{image_id}{ext}"
        if candidate.exists():
            return str(candidate)
    raise FileNotFoundError(f"Image not found for id={image_id} in {image_dir}")


weights = ResNet18_Weights.IMAGENET1K_V1
imagenet_mean = weights.transforms().mean
imagenet_std = weights.transforms().std

base_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])


class TrainWeightedDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row["label"]), float(row["sample_weight"])


class EvalDataset(Dataset):
    def __init__(self, dataframe, transform=None, has_label=True):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        if self.has_label:
            return image, int(row["label"]), str(row["id"])
        return image, str(row["id"])


def build_resnet18(num_classes=2, pretrained=True):
    model_weights = weights if pretrained else None
    model = models.resnet18(weights=model_weights)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    for param in model.parameters():
        param.requires_grad = True
    return model


def make_train_loader(dataframe, shuffle=True):
    return DataLoader(
        TrainWeightedDataset(dataframe, transform=base_transform),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )


def make_eval_loader(dataframe, has_label=True):
    return DataLoader(
        EvalDataset(dataframe, transform=base_transform, has_label=has_label),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )


def safe_auc(labels, probs):
    labels = np.asarray(labels)
    probs = np.asarray(probs)
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return np.nan
    return float(roc_auc_score(labels, probs))


def print_trainable_parameters(model, run_name):
    rows = []
    for name, param in model.named_parameters():
        if param.requires_grad:
            rows.append({
                "run": run_name,
                "name": name,
                "shape": str(tuple(param.shape)),
                "num_params": int(param.numel()),
            })
    df = pd.DataFrame(rows)
    print(f"[{run_name}] trainable parameter count:", int(df["num_params"].sum()) if len(df) else 0)
    display(df)
    return df


## 3. Load Labels and Reproduce Baseline Split

In [4]:
train_df = pd.read_csv(TRAIN_LABELS_PATH)
submission_template = pd.read_csv(SAMPLE_SUBMISSION_PATH)

expected_label_cols = ["id", "label"]
if train_df.columns.tolist() != expected_label_cols:
    raise ValueError(f"train_labels columns mismatch: {train_df.columns.tolist()}")
if len(train_df) != 1366:
    raise ValueError(f"train label row count must be 1366, got {len(train_df)}")
if train_df["id"].duplicated().any():
    raise ValueError("train_labels has duplicated ids")
if sorted(train_df["label"].unique().tolist()) != [0, 1]:
    raise ValueError("label must be binary 0/1")

train_df["id"] = train_df["id"].astype(str)
train_df["label"] = train_df["label"].astype(int)
train_df["path"] = train_df["id"].map(lambda image_id: image_path_from_id(TRAIN_DIR, image_id))

submission_template["id"] = submission_template["id"].astype(str)
test_df = submission_template[["id"]].copy()
test_df["path"] = test_df["id"].map(lambda image_id: image_path_from_id(TEST_DIR, image_id))

base_train_df, base_val_df = train_test_split(
    train_df,
    test_size=VAL_RATIO,
    random_state=SEED,
    stratify=train_df["label"],
)
base_train_df = base_train_df.reset_index(drop=True).copy()
base_val_df = base_val_df.reset_index(drop=True).copy()

split_df = pd.concat([
    base_train_df.assign(baseline_split="train"),
    base_val_df.assign(baseline_split="val"),
], ignore_index=True)

if len(split_df) != len(train_df) or split_df["id"].nunique() != len(train_df):
    raise ValueError("baseline split does not cover train_df exactly once")

split_df.to_csv(TABLE_DIR / "baseline_split_assignments.csv", index=False)

print("full label distribution:")
display(train_df["label"].value_counts().sort_index().rename_axis("label").reset_index(name="count"))
print("baseline split label distribution:")
display(pd.crosstab(split_df["baseline_split"], split_df["label"]))
print("saved baseline split assignments:", TABLE_DIR / "baseline_split_assignments.csv")


full label distribution:


,label,count
0,0,813
1,1,553


baseline split label distribution:


label,0,1
baseline_split,,
train,650,442
val,163,111


saved baseline split assignments: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_retrain_oof_top100_20260702_093712/tables/baseline_split_assignments.csv


## 4. Validate OOF File and Rebuild Best-AUC OOF if Needed

In [5]:
oof_config = json.loads(OOF_CONFIG_PATH.read_text())
oof_predict_checkpoint = oof_config.get("oof_predict_checkpoint")
print("OOF config checkpoint:", oof_predict_checkpoint)
print("OOF config n_splits:", oof_config.get("n_splits"), "epochs:", oof_config.get("epochs"))

raw_oof_df = pd.read_csv(OOF_SCORE_PATH)
raw_oof_df["id"] = raw_oof_df["id"].astype(str)
required_oof_cols = ["id", "path", "label", "fold", "oof_dusty_prob", "oof_pred", "mislabel_score_oof"]
missing_cols = [col for col in required_oof_cols if col not in raw_oof_df.columns]
if missing_cols:
    raise ValueError(f"OOF missing columns: {missing_cols}")
if len(raw_oof_df) != 1366:
    raise ValueError(f"OOF row count must be 1366, got {len(raw_oof_df)}")
if raw_oof_df["id"].nunique() != 1366:
    raise ValueError("OOF ids must be unique and cover 1366 samples")
if raw_oof_df[required_oof_cols].isna().any().any():
    raise ValueError("OOF required columns contain NaN")
if not raw_oof_df["oof_dusty_prob"].between(0, 1).all():
    raise ValueError("oof_dusty_prob must be in [0, 1]")

fold_assignment_df = pd.read_csv(OOF_FOLD_ASSIGNMENT_PATH)
fold_assignment_df["id"] = fold_assignment_df["id"].astype(str)
fold_assignment_df["label"] = fold_assignment_df["label"].astype(int)
fold_assignment_df["fold"] = fold_assignment_df["fold"].astype(int)
fold_assignment_df["path"] = fold_assignment_df["id"].map(lambda image_id: image_path_from_id(TRAIN_DIR, image_id))

if len(fold_assignment_df) != 1366 or fold_assignment_df["id"].nunique() != 1366:
    raise ValueError("fold assignments must cover 1366 unique ids")
if sorted(fold_assignment_df["fold"].unique().tolist()) != list(range(int(oof_config.get("n_splits", 5)))):
    raise ValueError("fold assignment values do not match OOF config")

checkpoint_audit_rows = []
fold_history_df = pd.read_csv(OOF_FOLD_HISTORY_PATH) if OOF_FOLD_HISTORY_PATH.exists() else pd.DataFrame()
for fold in sorted(fold_assignment_df["fold"].unique()):
    best_path = OOF_CHECKPOINT_DIR / f"fold{fold}_best_holdout_auc.pt"
    final_path = OOF_CHECKPOINT_DIR / f"fold{fold}_final.pt"
    best_epoch = np.nan
    best_auc = np.nan
    if best_path.exists():
        checkpoint = torch.load(best_path, map_location="cpu")
        best_epoch = checkpoint.get("epoch", np.nan)
        best_auc = checkpoint.get("best_holdout_auc", np.nan)
    history_best_epoch = np.nan
    history_best_auc = np.nan
    if not fold_history_df.empty:
        part = fold_history_df.loc[fold_history_df["fold"] == fold].copy()
        if len(part):
            idx = part["holdout_auc"].astype(float).idxmax()
            history_best_epoch = int(part.loc[idx, "epoch"])
            history_best_auc = float(part.loc[idx, "holdout_auc"])
    checkpoint_audit_rows.append({
        "fold": fold,
        "best_checkpoint_path": str(best_path),
        "best_checkpoint_exists": best_path.exists(),
        "final_checkpoint_exists": final_path.exists(),
        "checkpoint_best_epoch": best_epoch,
        "checkpoint_best_auc": best_auc,
        "history_best_epoch": history_best_epoch,
        "history_best_auc": history_best_auc,
    })

checkpoint_audit_df = pd.DataFrame(checkpoint_audit_rows)
checkpoint_audit_df.to_csv(TABLE_DIR / "oof_checkpoint_audit.csv", index=False)
display(checkpoint_audit_df)

best_checkpoint_ready = bool(checkpoint_audit_df["best_checkpoint_exists"].all())
oof_is_best_auc = oof_predict_checkpoint == EXPECTED_OOF_CHECKPOINT
print("OOF file claims best-AUC checkpoint:", oof_is_best_auc)
print("best checkpoint files ready:", best_checkpoint_ready)


def predict_labeled_dataframe(model, dataframe):
    loader = make_eval_loader(dataframe, has_label=True)
    model.eval()
    rows = []
    with torch.no_grad():
        for images, labels, image_ids in loader:
            images = images.to(device, non_blocking=PIN_MEMORY)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            preds = logits.argmax(dim=1).detach().cpu().numpy()
            labels_np = labels.detach().cpu().numpy()
            for image_id, label, pred, prob in zip(image_ids, labels_np, preds, probs):
                rows.append({
                    "id": str(image_id),
                    "label": int(label),
                    "oof_pred": int(pred),
                    "oof_dusty_prob": float(prob),
                })
    return pd.DataFrame(rows)


def rebuild_oof_from_best_checkpoints():
    pred_parts = []
    for fold in sorted(fold_assignment_df["fold"].unique()):
        checkpoint_path = OOF_CHECKPOINT_DIR / f"fold{fold}_best_holdout_auc.pt"
        if not checkpoint_path.exists():
            raise FileNotFoundError(checkpoint_path)
        holdout_df = fold_assignment_df.loc[fold_assignment_df["fold"] == fold].copy()
        model = build_resnet18(num_classes=2, pretrained=False).to(device)
        checkpoint = torch.load(checkpoint_path, map_location=device)
        load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=False)
        if load_result.missing_keys or load_result.unexpected_keys:
            raise ValueError(
                f"fold {fold} checkpoint key mismatch: "
                f"missing={load_result.missing_keys}, unexpected={load_result.unexpected_keys}"
            )
        pred_df = predict_labeled_dataframe(model, holdout_df)
        pred_df["fold"] = fold
        pred_df["checkpoint_path"] = str(checkpoint_path)
        pred_df["checkpoint_epoch"] = checkpoint.get("epoch", np.nan)
        pred_df["checkpoint_best_holdout_auc"] = checkpoint.get("best_holdout_auc", np.nan)
        pred_parts.append(pred_df)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    rebuilt = pd.concat(pred_parts, ignore_index=True)
    rebuilt = fold_assignment_df[["id", "path", "label", "fold"]].merge(
        rebuilt.drop(columns=["label", "fold"]),
        on="id",
        how="left",
    )
    rebuilt["mislabel_score_oof"] = np.where(
        rebuilt["label"].astype(int) == 0,
        rebuilt["oof_dusty_prob"].astype(float),
        1.0 - rebuilt["oof_dusty_prob"].astype(float),
    )
    for col in ["dup_score", "mislabel_score_v2_add"]:
        if col in raw_oof_df.columns:
            rebuilt = rebuilt.merge(raw_oof_df[["id", col]], on="id", how="left")
    return rebuilt


if oof_is_best_auc:
    oof_df = raw_oof_df.copy()
    oof_source = "existing_best_auc_oof_file"
elif REBUILD_BEST_AUC_OOF_IF_NEEDED and best_checkpoint_ready:
    print("Existing OOF file is not best-AUC based. Rebuilding OOF predictions from fold best checkpoints...")
    oof_df = rebuild_oof_from_best_checkpoints()
    oof_source = "rebuilt_from_best_auc_checkpoints"
    rebuilt_path = TABLE_DIR / "oof_mislabel_scores_best_auc_rebuilt.csv"
    oof_df.to_csv(rebuilt_path, index=False)
    print("saved rebuilt best-AUC OOF:", rebuilt_path)
else:
    oof_df = raw_oof_df.copy()
    oof_source = "existing_non_best_oof_file"

if REQUIRE_BEST_AUC_OOF and oof_source == "existing_non_best_oof_file":
    raise ValueError(
        "OOF file is not best-AUC based and best checkpoints could not be used to rebuild it. "
        "Set REQUIRE_BEST_AUC_OOF=False only for a diagnostic dry run."
    )

print("OOF source used by this notebook:", oof_source)


OOF config checkpoint: final
OOF config n_splits: 5 epochs: 10


,fold,best_checkpoint_path,best_checkpoint_exists,final_checkpoint_exists,checkpoint_best_epoch,checkpoint_best_auc,history_best_epoch,history_best_auc
0,0,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True,True,9,0.774001,9,0.774001
1,1,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True,True,10,0.751813,10,0.751813
2,2,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True,True,7,0.800725,7,0.800725
3,3,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True,True,1,0.820209,1,0.820209
4,4,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True,True,4,0.794406,4,0.794406


OOF file claims best-AUC checkpoint: False
best checkpoint files ready: True
Existing OOF file is not best-AUC based. Rebuilding OOF predictions from fold best checkpoints...


/home/thkim0/venv/v1/lib/python3.10/site-packages/torch/nn/modules/conv.py:459: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,


saved rebuilt best-AUC OOF: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_retrain_oof_top100_20260702_093712/tables/oof_mislabel_scores_best_auc_rebuilt.csv
OOF source used by this notebook: rebuilt_from_best_auc_checkpoints


## 5. Compute Mislabel Score and Compare with v2

In [6]:
oof_df = oof_df.copy()
oof_df["id"] = oof_df["id"].astype(str)
oof_df["label"] = oof_df["label"].astype(int)
oof_df["fold"] = oof_df["fold"].astype(int)
oof_df["oof_dusty_prob"] = oof_df["oof_dusty_prob"].astype(float)

oof_df["mislabel_score_oof_recomputed"] = np.where(
    oof_df["label"] == 0,
    oof_df["oof_dusty_prob"],
    1.0 - oof_df["oof_dusty_prob"],
)
score_diff = (oof_df["mislabel_score_oof"].astype(float) - oof_df["mislabel_score_oof_recomputed"]).abs().max()
print("max abs diff for mislabel_score_oof recomputation:", score_diff)
if score_diff > 1e-6:
    raise ValueError("mislabel_score_oof does not match recomputed formula")
oof_df["mislabel_score_oof"] = oof_df["mislabel_score_oof_recomputed"]
oof_df = oof_df.drop(columns=["mislabel_score_oof_recomputed"])

if "mislabel_score_v2_add" not in oof_df.columns and V2_DETAIL_PATH is not None:
    v2_df = pd.read_csv(V2_DETAIL_PATH)
    v2_df["id"] = v2_df["id"].astype(str)
    merge_cols = ["id"] + [col for col in ["mislabel_score_v2_add", "dup_score"] if col in v2_df.columns]
    oof_df = oof_df.merge(v2_df[merge_cols], on="id", how="left")

for col in ["dup_score", "mislabel_score_v2_add"]:
    if col not in oof_df.columns:
        oof_df[col] = 0.0
    oof_df[col] = oof_df[col].fillna(0.0).astype(float)

score_df = split_df[["id", "path", "label", "baseline_split"]].merge(
    oof_df[["id", "fold", "oof_dusty_prob", "oof_pred", "mislabel_score_oof", "dup_score", "mislabel_score_v2_add"]],
    on="id",
    how="left",
)
if score_df[["fold", "oof_dusty_prob", "mislabel_score_oof"]].isna().any().any():
    raise ValueError("OOF scores did not merge onto every train label")

score_df["mislabel_rank_oof"] = score_df["mislabel_score_oof"].rank(method="first", ascending=False).astype(int)
score_df["is_oof_top100"] = score_df["mislabel_rank_oof"] <= TOPK_REMOVE
score_df.to_csv(TABLE_DIR / "trackA_retrain_score_table.csv", index=False)

validation_summary = pd.DataFrame([
    {"check": "oof_rows", "value": len(oof_df)},
    {"check": "oof_unique_ids", "value": oof_df["id"].nunique()},
    {"check": "oof_source", "value": oof_source},
    {"check": "mislabel_score_recompute_max_abs_diff", "value": score_diff},
])
validation_summary.to_csv(TABLE_DIR / "oof_validation_summary.csv", index=False)
display(validation_summary)

compare_rows = []
if "mislabel_score_v2_add" in score_df.columns:
    top100_oof_ids = set(score_df.nsmallest(TOPK_REMOVE, "mislabel_rank_oof")["id"])
    top100_v2_ids = set(score_df.sort_values("mislabel_score_v2_add", ascending=False).head(TOPK_REMOVE)["id"])
    overlap = len(top100_oof_ids & top100_v2_ids)
    pearson = score_df[["mislabel_score_oof", "mislabel_score_v2_add"]].corr(method="pearson").iloc[0, 1]
    spearman = score_df[["mislabel_score_oof", "mislabel_score_v2_add"]].corr(method="spearman").iloc[0, 1]
    compare_rows.append({
        "topk": TOPK_REMOVE,
        "oof_v2_overlap": overlap,
        "oof_v2_overlap_ratio": overlap / TOPK_REMOVE,
        "pearson_corr": pearson,
        "spearman_corr": spearman,
    })

compare_df = pd.DataFrame(compare_rows)
compare_df.to_csv(TABLE_DIR / "oof_vs_v2_compare.csv", index=False)
print("OOF vs v2 comparison:")
display(compare_df)
print("saved score table:", TABLE_DIR / "trackA_retrain_score_table.csv")


max abs diff for mislabel_score_oof recomputation: 0.0


,check,value
0,oof_rows,1366
1,oof_unique_ids,1366
2,oof_source,rebuilt_from_best_auc_checkpoints
3,mislabel_score_recompute_max_abs_diff,0.0


OOF vs v2 comparison:


,topk,oof_v2_overlap,oof_v2_overlap_ratio,pearson_corr,spearman_corr
0,100,21,0.21,0.230292,0.181072


saved score table: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_retrain_oof_top100_20260702_093712/tables/trackA_retrain_score_table.csv


## 6. Top-K Split Diagnostics and Top100 Image Grid

In [7]:
topk_distribution_rows = []
for topk in TOPK_REPORTS:
    part = score_df.sort_values("mislabel_score_oof", ascending=False).head(topk).copy()
    for (split_name, label), count in part.groupby(["baseline_split", "label"]).size().items():
        topk_distribution_rows.append({
            "topk": topk,
            "baseline_split": split_name,
            "label": int(label),
            "count": int(count),
        })

topk_distribution_df = pd.DataFrame(topk_distribution_rows)
topk_distribution_df.to_csv(TABLE_DIR / "topk_baseline_split_label_distribution.csv", index=False)
print("top50/top100/top150 baseline train/val distribution:")
display(topk_distribution_df)

def compute_dup_sample_weight(dup_score):
    dup_score = np.asarray(dup_score, dtype=float)
    dup_score = np.clip(dup_score, 0.0, 1.0)
    weight = 1.0 - (1.0 - DUP_WEIGHT_MIN) * np.power(dup_score, DUP_WEIGHT_POWER)
    return np.clip(weight, DUP_WEIGHT_MIN, 1.0)

score_df["sample_weight_before_removal"] = compute_dup_sample_weight(score_df["dup_score"].to_numpy())

top100_df = score_df.sort_values("mislabel_score_oof", ascending=False).head(TOPK_REMOVE).copy()
top100_df["removed_from_train"] = top100_df["baseline_split"].eq("train")
top100_path = TABLE_DIR / "removed_mislabel_top100.csv"
top100_df[[
    "id",
    "path",
    "label",
    "baseline_split",
    "fold",
    "mislabel_score_oof",
    "mislabel_rank_oof",
    "dup_score",
    "sample_weight_before_removal",
    "removed_from_train",
]].to_csv(top100_path, index=False)
print("saved top100 candidates:", top100_path)

grid_cols = 10
grid_rows = 10
fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(22, 22))
for ax, (_, row) in zip(axes.ravel(), top100_df.iterrows()):
    image = Image.open(row["path"]).convert("RGB")
    ax.imshow(image)
    title_line1 = f"#{int(row['mislabel_rank_oof'])} {row['id']}"
    title_line2 = (
        f"s={row['mislabel_score_oof']:.3f} "
        f"y={int(row['label'])} p={int(row['oof_pred'])} {row['baseline_split']}"
    )
    ax.set_title(f"{title_line1}\n{title_line2}", fontsize=6)
    ax.axis("off")
for ax in axes.ravel()[len(top100_df):]:
    ax.axis("off")
fig.tight_layout()
grid_path = VIS_DIR / "oof_mislabel_top100_grid.png"
fig.savefig(grid_path, dpi=180)
plt.close(fig)
print("saved top100 grid:", grid_path)

display(top100_df[["id", "label", "baseline_split", "mislabel_score_oof", "dup_score", "sample_weight_before_removal"]].head(10))


top50/top100/top150 baseline train/val distribution:


,topk,baseline_split,label,count
0,50,train,0,9
1,50,train,1,28
2,50,val,0,4
3,50,val,1,9
4,100,train,0,30
5,100,train,1,54
6,100,val,0,5
7,100,val,1,11
8,150,train,0,43
9,150,train,1,78


saved top100 candidates: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_retrain_oof_top100_20260702_093712/tables/removed_mislabel_top100.csv
saved top100 grid: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_retrain_oof_top100_20260702_093712/visualizations/oof_mislabel_top100_grid.png


,id,label,baseline_split,mislabel_score_oof,dup_score,sample_weight_before_removal
951,train_01241,1,train,0.999999,0.0000,1.00000
483,train_00509,1,train,0.999996,0.0625,0.96875
867,train_00613,1,train,0.999994,0.0000,1.00000
503,train_00238,1,train,0.999976,0.0000,1.00000
822,train_00394,0,train,0.999969,0.0000,1.00000
115,train_00342,1,train,0.999944,0.0000,1.00000
938,train_00356,1,train,0.999926,0.0000,1.00000
628,train_01135,1,train,0.999917,1.0000,0.50000
972,train_01177,1,train,0.999904,0.2500,0.87500
1346,train_00199,1,val,0.999897,0.0000,1.00000


In [8]:
# OOF top-k fold concentration diagnostics
# Purpose: check whether high mislabel_score_oof samples are concentrated in a specific OOF fold.
from pathlib import Path
import pandas as pd

if "score_df" in globals():
    fold_count_source = score_df.copy()
    source_name = "score_df"
elif "oof_df" in globals():
    fold_count_source = oof_df.copy()
    source_name = "oof_df"
else:
    fallback_oof_path = globals().get(
        "OOF_SCORE_PATH",
        Path("/home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackB_oof_mislabel/oof_mislabel_scores.csv"),
    )
    fold_count_source = pd.read_csv(fallback_oof_path)
    source_name = str(fallback_oof_path)

required_cols = {"fold", "mislabel_score_oof"}
missing_cols = required_cols - set(fold_count_source.columns)
if missing_cols:
    raise ValueError(f"fold concentration source is missing columns: {sorted(missing_cols)}")

fold_count_source["fold"] = fold_count_source["fold"].astype(int)
fold_count_source["mislabel_score_oof"] = fold_count_source["mislabel_score_oof"].astype(float)
all_folds = sorted(fold_count_source["fold"].unique().tolist())

print("fold concentration source:", source_name)
print("rows:", len(fold_count_source))
print("all fold totals:")
all_fold_counts = fold_count_source["fold"].value_counts().reindex(all_folds, fill_value=0).sort_index()
for fold, count in all_fold_counts.items():
    print(f"fold{fold}: {int(count)}")

summary_rows = []
for k in [50, 100, 150]:
    top = fold_count_source.sort_values("mislabel_score_oof", ascending=False).head(k).copy()
    fold_counts = top["fold"].value_counts().reindex(all_folds, fill_value=0).sort_index()

    print(f"\ntop{k} fold distribution")
    for fold, count in fold_counts.items():
        print(f"fold{fold}: {int(count)}")

    for fold, count in fold_counts.items():
        summary_rows.append({
            "topk": k,
            "fold": int(fold),
            "count": int(count),
            "ratio": float(count / k),
        })

fold_concentration_df = pd.DataFrame(summary_rows)
if "TABLE_DIR" in globals():
    fold_concentration_path = TABLE_DIR / "topk_oof_fold_concentration.csv"
    fold_concentration_df.to_csv(fold_concentration_path, index=False)
    print("\nsaved fold concentration table:", fold_concentration_path)

display(fold_concentration_df)


fold concentration source: score_df
rows: 1366
all fold totals:
fold0: 274
fold1: 273
fold2: 273
fold3: 273
fold4: 273

top50 fold distribution
fold0: 19
fold1: 9
fold2: 17
fold3: 0
fold4: 5

top100 fold distribution
fold0: 30
fold1: 25
fold2: 30
fold3: 1
fold4: 14

top150 fold distribution
fold0: 43
fold1: 36
fold2: 40
fold3: 4
fold4: 27

saved fold concentration table: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_retrain_oof_top100_20260702_093712/tables/topk_oof_fold_concentration.csv


,topk,fold,count,ratio
0,50,0,19,0.380000
1,50,1,9,0.180000
2,50,2,17,0.340000
3,50,3,0,0.000000
4,50,4,5,0.100000
5,100,0,30,0.300000
6,100,1,25,0.250000
7,100,2,30,0.300000
8,100,3,1,0.010000
9,100,4,14,0.140000


## 7. Build R1/R2 Train Sets and Sample Weights

In [9]:
top100_ids = set(top100_df["id"])
train_candidates_before = score_df.loc[score_df["baseline_split"] == "train"].copy()
val_all_df = score_df.loc[score_df["baseline_split"] == "val"].copy()
val_without_top100_df = val_all_df.loc[~val_all_df["id"].isin(top100_ids)].copy()
val_top100_only_df = val_all_df.loc[val_all_df["id"].isin(top100_ids)].copy()

train_removed_df = train_candidates_before.loc[train_candidates_before["id"].isin(top100_ids)].copy()
train_candidates_after = train_candidates_before.loc[~train_candidates_before["id"].isin(top100_ids)].copy()

class_balance_rows = []
for name, df in [
    ("train_before_removal", train_candidates_before),
    ("removed_from_train_top100", train_removed_df),
    ("train_after_removal", train_candidates_after),
    ("val_all", val_all_df),
    ("val_without_top100", val_without_top100_df),
    ("val_top100_only", val_top100_only_df),
]:
    counts = df["label"].value_counts().sort_index()
    for label in [0, 1]:
        class_balance_rows.append({
            "subset": name,
            "label": label,
            "count": int(counts.get(label, 0)),
            "total": int(len(df)),
        })
class_balance_df = pd.DataFrame(class_balance_rows)
class_balance_df.to_csv(TABLE_DIR / "class_balance_before_after.csv", index=False)
print("class balance before/after top100 removal:")
display(class_balance_df)


def build_variant_train_df(use_dup_weight):
    df = train_candidates_after.copy()
    if use_dup_weight:
        df["sample_weight"] = compute_dup_sample_weight(df["dup_score"].to_numpy())
    else:
        df["sample_weight"] = 1.0
    return df.reset_index(drop=True)

r1_train_df = build_variant_train_df(use_dup_weight=False)
r2_train_df = build_variant_train_df(use_dup_weight=True)

r1_train_df.to_csv(TABLE_DIR / "R1_train_rows.csv", index=False)
r2_train_df.to_csv(TABLE_DIR / "R2_train_rows.csv", index=False)
train_removed_df.to_csv(TABLE_DIR / "train_removed_oof_top100_rows.csv", index=False)

weight_stat_rows = []
for run_name, df in [("R1", r1_train_df), ("R2", r2_train_df)]:
    sample_weight = df["sample_weight"].astype(float)
    dup_score = df["dup_score"].astype(float)
    weight_stat_rows.append({
        "run": run_name,
        "rows": len(df),
        "sum_sample_weight": float(sample_weight.sum()),
        "mean_sample_weight": float(sample_weight.mean()),
        "min_sample_weight": float(sample_weight.min()),
        "q25_sample_weight": float(sample_weight.quantile(0.25)),
        "q50_sample_weight": float(sample_weight.quantile(0.50)),
        "q75_sample_weight": float(sample_weight.quantile(0.75)),
        "max_sample_weight": float(sample_weight.max()),
        "min_dup_score": float(dup_score.min()),
        "mean_dup_score": float(dup_score.mean()),
        "max_dup_score": float(dup_score.max()),
        "q90_dup_score": float(dup_score.quantile(0.90)),
        "q99_dup_score": float(dup_score.quantile(0.99)),
    })
weight_stats_df = pd.DataFrame(weight_stat_rows)
weight_stats_df.to_csv(TABLE_DIR / "sample_weight_stats.csv", index=False)
print("sample weight and dup score stats:")
display(weight_stats_df)


class balance before/after top100 removal:


,subset,label,count,total
0,train_before_removal,0,650,1092
1,train_before_removal,1,442,1092
2,removed_from_train_top100,0,30,84
3,removed_from_train_top100,1,54,84
4,train_after_removal,0,620,1008
5,train_after_removal,1,388,1008
6,val_all,0,163,274
7,val_all,1,111,274
8,val_without_top100,0,158,258
9,val_without_top100,1,100,258


sample weight and dup score stats:


,run,rows,sum_sample_weight,mean_sample_weight,min_sample_weight,q25_sample_weight,q50_sample_weight,q75_sample_weight,max_sample_weight,min_dup_score,mean_dup_score,max_dup_score,q90_dup_score,q99_dup_score
0,R1,1008,1008.00000,1.000000,1.0,1.0,1.0,1.0,1.0,0.0,0.085627,1.0,0.25,1.0
1,R2,1008,964.84375,0.957186,0.5,1.0,1.0,1.0,1.0,0.0,0.085627,1.0,0.25,1.0


## 8. Training and Evaluation Helpers

In [10]:
def binary_outputs(labels_tensor, logits_tensor):
    probs = torch.softmax(logits_tensor, dim=1)[:, 1]
    preds = logits_tensor.argmax(dim=1)
    return (
        labels_tensor.detach().cpu().numpy(),
        preds.detach().cpu().numpy(),
        probs.detach().cpu().numpy(),
    )


def train_one_epoch_weighted(model, dataloader, optimizer):
    model.train()
    loss_fn = nn.CrossEntropyLoss(reduction="none")
    total_weighted_loss = 0.0
    total_weight = 0.0
    all_labels, all_preds, all_probs = [], [], []

    for images, labels, sample_weights in dataloader:
        images = images.to(device, non_blocking=PIN_MEMORY)
        labels = labels.to(device, non_blocking=PIN_MEMORY)
        sample_weights = sample_weights.to(device, non_blocking=PIN_MEMORY).float()

        logits = model(images)
        loss_raw = loss_fn(logits, labels)
        loss = (loss_raw * sample_weights).sum() / sample_weights.sum().clamp_min(1e-8)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_weight = float(sample_weights.sum().item())
        total_weighted_loss += float(loss.item()) * batch_weight
        total_weight += batch_weight

        labels_np, preds_np, probs_np = binary_outputs(labels, logits)
        all_labels.extend(labels_np.tolist())
        all_preds.extend(preds_np.tolist())
        all_probs.extend(probs_np.tolist())

    return {
        "loss": total_weighted_loss / max(total_weight, 1e-8),
        "acc": float(accuracy_score(all_labels, all_preds)),
        "auc": safe_auc(all_labels, all_probs),
        "sum_sample_weight": total_weight,
        "mean_sample_weight": total_weight / max(len(dataloader.dataset), 1),
    }


def evaluate_labeled(model, dataframe, subset_name):
    if len(dataframe) == 0:
        return {
            "subset": subset_name,
            "n": 0,
            "label0": 0,
            "label1": 0,
            "loss": np.nan,
            "acc": np.nan,
            "auc": np.nan,
            "mean_dusty_prob": np.nan,
        }
    loader = make_eval_loader(dataframe, has_label=True)
    loss_fn = nn.CrossEntropyLoss(reduction="sum")
    model.eval()
    total_loss = 0.0
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for images, labels, _ in loader:
            images = images.to(device, non_blocking=PIN_MEMORY)
            labels = labels.to(device, non_blocking=PIN_MEMORY)
            logits = model(images)
            total_loss += float(loss_fn(logits, labels).item())
            labels_np, preds_np, probs_np = binary_outputs(labels, logits)
            all_labels.extend(labels_np.tolist())
            all_preds.extend(preds_np.tolist())
            all_probs.extend(probs_np.tolist())
    label_counts = pd.Series(all_labels).value_counts().to_dict()
    return {
        "subset": subset_name,
        "n": len(all_labels),
        "label0": int(label_counts.get(0, 0)),
        "label1": int(label_counts.get(1, 0)),
        "loss": total_loss / len(all_labels),
        "acc": float(accuracy_score(all_labels, all_preds)),
        "auc": safe_auc(all_labels, all_probs),
        "mean_dusty_prob": float(np.mean(all_probs)),
    }


def evaluate_all_val_subsets(model, run_name, epoch=None):
    rows = []
    for subset_name, df in [
        ("val_all", val_all_df),
        ("val_without_top100", val_without_top100_df),
        ("val_top100_only", val_top100_only_df),
    ]:
        row = evaluate_labeled(model, df, subset_name)
        row["run"] = run_name
        row["epoch"] = epoch
        rows.append(row)
    return pd.DataFrame(rows)


def predict_test_submission(model, result_path):
    loader = make_eval_loader(test_df, has_label=False)
    model.eval()
    rows = []
    with torch.no_grad():
        for images, image_ids in loader:
            images = images.to(device, non_blocking=PIN_MEMORY)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            for image_id, prob in zip(image_ids, probs):
                rows.append({"id": str(image_id), "dusty_prob": float(prob)})
    pred_df = pd.DataFrame(rows)
    result_df = submission_template[["id"]].merge(pred_df, on="id", how="left")
    if result_df["dusty_prob"].isna().any():
        raise ValueError("submission has missing predictions")
    if result_df["id"].tolist() != submission_template["id"].tolist():
        raise ValueError("submission id order mismatch")
    if not result_df["dusty_prob"].between(0, 1).all():
        raise ValueError("dusty_prob must be in [0, 1]")
    result_df.to_csv(result_path, index=False)
    return result_df


def load_checkpoint_strict(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=False)
    print("checkpoint:", checkpoint_path)
    print("missing keys:", load_result.missing_keys)
    print("unexpected keys:", load_result.unexpected_keys)
    if load_result.missing_keys or load_result.unexpected_keys:
        raise ValueError("checkpoint key mismatch")
    return checkpoint


## 9. R0 Baseline Checkpoint Inference

In [11]:
all_metric_rows = []
submission_paths = {}

r0_model = build_resnet18(num_classes=2, pretrained=False).to(device)
r0_checkpoint = load_checkpoint_strict(r0_model, BASELINE_CHECKPOINT_PATH)
print("R0 loaded epoch:", r0_checkpoint.get("epoch"))
print("R0 loaded best_val_auc:", r0_checkpoint.get("best_val_auc"))
r0_trainable_df = print_trainable_parameters(r0_model, "R0_loaded_baseline")
r0_trainable_df.to_csv(TABLE_DIR / "R0_trainable_parameters.csv", index=False)

r0_metrics_df = evaluate_all_val_subsets(r0_model, "R0", epoch=r0_checkpoint.get("epoch"))
all_metric_rows.append(r0_metrics_df)
print("R0 validation metrics:")
display(r0_metrics_df)

r0_submission_path = SUBMISSION_DIR / "submissionA_R0_baseline.csv"
r0_submission = predict_test_submission(r0_model, r0_submission_path)
submission_paths["R0"] = r0_submission_path
print("saved R0 submission:", r0_submission_path)
display(r0_submission["dusty_prob"].describe())

del r0_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


checkpoint: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/model/base0_resnet18_finetune_best.pt
missing keys: []
unexpected keys: []
R0 loaded epoch: 5
R0 loaded best_val_auc: 0.7937323826894379
[R0_loaded_baseline] trainable parameter count: 11177538


,run,name,shape,num_params
0,R0_loaded_baseline,conv1.weight,"(64, 3, 7, 7)",9408
1,R0_loaded_baseline,bn1.weight,"(64,)",64
2,R0_loaded_baseline,bn1.bias,"(64,)",64
3,R0_loaded_baseline,layer1.0.conv1.weight,"(64, 64, 3, 3)",36864
4,R0_loaded_baseline,layer1.0.bn1.weight,"(64,)",64
...,...,...,...,...
57,R0_loaded_baseline,layer4.1.conv2.weight,"(512, 512, 3, 3)",2359296
58,R0_loaded_baseline,layer4.1.bn2.weight,"(512,)",512
59,R0_loaded_baseline,layer4.1.bn2.bias,"(512,)",512
60,R0_loaded_baseline,fc.weight,"(2, 512)",1024


R0 validation metrics:


,subset,n,label0,label1,loss,acc,auc,mean_dusty_prob,run,epoch
0,val_all,274,163,111,0.851076,0.740876,0.793732,0.346828,R0,5
1,val_without_top100,258,158,100,0.660200,0.782946,0.842278,0.346472,R0,5
2,val_top100_only,16,5,11,3.928942,0.062500,0.018182,0.352573,R0,5


saved R0 submission: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_retrain_oof_top100_20260702_093712/submissions/submissionA_R0_baseline.csv


count    4.680000e+02
mean     3.413103e-01
std      3.844580e-01
min      2.560436e-08
25%      9.604319e-03
50%      1.284836e-01
75%      7.346280e-01
max      9.999746e-01
Name: dusty_prob, dtype: float64

## 10. R1/R2 Retraining

In [12]:
def train_variant(run_name, train_part_df, use_dup_weight):
    set_seed(SEED)
    run_dir = CHECKPOINT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = run_dir / f"{run_name}_best_val_all_auc.pt"
    history_path = TABLE_DIR / f"{run_name}_history.csv"
    submission_path = SUBMISSION_DIR / f"submissionA_{run_name}.csv"

    model = build_resnet18(num_classes=2, pretrained=True).to(device)
    trainable_df = print_trainable_parameters(model, run_name)
    trainable_df.to_csv(TABLE_DIR / f"{run_name}_trainable_parameters.csv", index=False)

    if summary is not None:
        try:
            summary(
                model,
                input_size=(BATCH_SIZE, 3, 224, 224),
                col_names=("input_size", "output_size", "num_params", "trainable"),
                depth=2,
                device=device,
            )
        except Exception as e:
            print("torchinfo.summary failed:", repr(e))

    train_loader = make_train_loader(train_part_df, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    print(f"[{run_name}] train rows:", len(train_part_df))
    print(f"[{run_name}] use_dup_weight:", use_dup_weight)
    print(f"[{run_name}] effective sample size sum:", float(train_part_df["sample_weight"].sum()))
    print(f"[{run_name}] effective sample weight mean:", float(train_part_df["sample_weight"].mean()))

    history_rows = []
    best_val_all_auc = -np.inf
    best_epoch = None

    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()
        train_metrics = train_one_epoch_weighted(model, train_loader, optimizer)
        val_metrics_df = evaluate_all_val_subsets(model, run_name, epoch=epoch)
        val_all_auc = val_metrics_df.loc[val_metrics_df["subset"] == "val_all", "auc"].iloc[0]
        elapsed = time.time() - start_time

        row = {
            "run": run_name,
            "epoch": epoch,
            "train_rows": len(train_part_df),
            "use_dup_weight": use_dup_weight,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["acc"],
            "train_auc": train_metrics["auc"],
            "train_sum_sample_weight": train_metrics["sum_sample_weight"],
            "train_mean_sample_weight": train_metrics["mean_sample_weight"],
            "elapsed_sec": elapsed,
        }
        for _, metric_row in val_metrics_df.iterrows():
            prefix = metric_row["subset"]
            row[f"{prefix}_n"] = metric_row["n"]
            row[f"{prefix}_loss"] = metric_row["loss"]
            row[f"{prefix}_acc"] = metric_row["acc"]
            row[f"{prefix}_auc"] = metric_row["auc"]
        history_rows.append(row)

        print(
            f"{run_name} epoch={epoch:02d}/{EPOCHS} | "
            f"train_loss={train_metrics['loss']:.4f} train_auc={train_metrics['auc']:.4f} | "
            f"val_all_auc={row['val_all_auc']:.4f} "
            f"val_without_top100_auc={row['val_without_top100_auc']:.4f} "
            f"val_top100_only_auc={row['val_top100_only_auc']:.4f} | {elapsed:.1f}s"
        )

        if np.isfinite(val_all_auc) and val_all_auc > best_val_all_auc:
            best_val_all_auc = float(val_all_auc)
            best_epoch = epoch
            torch.save(
                {
                    "run": run_name,
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_all_auc": best_val_all_auc,
                    "config": config,
                    "use_dup_weight": use_dup_weight,
                    "removed_top100_ids": sorted(top100_ids),
                },
                checkpoint_path,
            )
            print("  -> saved best checkpoint:", checkpoint_path)

    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(history_path, index=False)

    checkpoint = load_checkpoint_strict(model, checkpoint_path)
    final_metrics_df = evaluate_all_val_subsets(model, run_name, epoch=checkpoint.get("epoch"))
    submission_df = predict_test_submission(model, submission_path)

    print(f"[{run_name}] best epoch:", best_epoch, "best_val_all_auc:", best_val_all_auc)
    print(f"[{run_name}] saved history:", history_path)
    print(f"[{run_name}] saved submission:", submission_path)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "run": run_name,
        "checkpoint_path": checkpoint_path,
        "history_path": history_path,
        "submission_path": submission_path,
        "history_df": history_df,
        "final_metrics_df": final_metrics_df,
        "submission_df": submission_df,
    }


variant_results = {}
if RUN_R1:
    variant_results["R1"] = train_variant("R1_remove_oof_top100", r1_train_df, use_dup_weight=False)
    all_metric_rows.append(variant_results["R1"]["final_metrics_df"])
    submission_paths["R1"] = variant_results["R1"]["submission_path"]
else:
    print("RUN_R1=False; skip R1 retraining")

if RUN_R2:
    variant_results["R2"] = train_variant("R2_remove_oof_top100_dup_weight", r2_train_df, use_dup_weight=True)
    all_metric_rows.append(variant_results["R2"]["final_metrics_df"])
    submission_paths["R2"] = variant_results["R2"]["submission_path"]
else:
    print("RUN_R2=False; skip R2 retraining")

val_metrics_by_subset_df = pd.concat(all_metric_rows, ignore_index=True)
val_metrics_by_subset_df.to_csv(TABLE_DIR / "val_metrics_by_subset.csv", index=False)
print("saved val subset metrics:", TABLE_DIR / "val_metrics_by_subset.csv")
display(val_metrics_by_subset_df)


[R1_remove_oof_top100] trainable parameter count: 11177538


,run,name,shape,num_params
0,R1_remove_oof_top100,conv1.weight,"(64, 3, 7, 7)",9408
1,R1_remove_oof_top100,bn1.weight,"(64,)",64
2,R1_remove_oof_top100,bn1.bias,"(64,)",64
3,R1_remove_oof_top100,layer1.0.conv1.weight,"(64, 64, 3, 3)",36864
4,R1_remove_oof_top100,layer1.0.bn1.weight,"(64,)",64
...,...,...,...,...
57,R1_remove_oof_top100,layer4.1.conv2.weight,"(512, 512, 3, 3)",2359296
58,R1_remove_oof_top100,layer4.1.bn2.weight,"(512,)",512
59,R1_remove_oof_top100,layer4.1.bn2.bias,"(512,)",512
60,R1_remove_oof_top100,fc.weight,"(2, 512)",1024


[R1_remove_oof_top100] train rows: 1008
[R1_remove_oof_top100] use_dup_weight: False
[R1_remove_oof_top100] effective sample size sum: 1008.0
[R1_remove_oof_top100] effective sample weight mean: 1.0
R1_remove_oof_top100 epoch=01/10 | train_loss=0.5595 train_auc=0.7809 | val_all_auc=0.7858 val_without_top100_auc=0.8385 val_top100_only_auc=0.0182 | 6.8s
  -> saved best checkpoint: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_retrain_oof_top100_20260702_093712/checkpoints/R1_remove_oof_top100/R1_remove_oof_top100_best_val_all_auc.pt
R1_remove_oof_top100 epoch=02/10 | train_loss=0.2136 train_auc=0.9759 | val_all_auc=0.7915 val_without_top100_auc=0.8450 val_top100_only_auc=0.0727 | 6.7s
  -> saved best checkpoint: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_retrain_oof_top100_20260702_093712/checkpoints/R1_remove_oof_top100/R1_remove_oof_top100_best_val_all_auc.pt
R1_remove_oof_top100 epoch=03/10 | train_loss=0.1068 train_auc=0.9945 | val_all

,run,name,shape,num_params
0,R2_remove_oof_top100_dup_weight,conv1.weight,"(64, 3, 7, 7)",9408
1,R2_remove_oof_top100_dup_weight,bn1.weight,"(64,)",64
2,R2_remove_oof_top100_dup_weight,bn1.bias,"(64,)",64
3,R2_remove_oof_top100_dup_weight,layer1.0.conv1.weight,"(64, 64, 3, 3)",36864
4,R2_remove_oof_top100_dup_weight,layer1.0.bn1.weight,"(64,)",64
...,...,...,...,...
57,R2_remove_oof_top100_dup_weight,layer4.1.conv2.weight,"(512, 512, 3, 3)",2359296
58,R2_remove_oof_top100_dup_weight,layer4.1.bn2.weight,"(512,)",512
59,R2_remove_oof_top100_dup_weight,layer4.1.bn2.bias,"(512,)",512
60,R2_remove_oof_top100_dup_weight,fc.weight,"(2, 512)",1024


[R2_remove_oof_top100_dup_weight] train rows: 1008
[R2_remove_oof_top100_dup_weight] use_dup_weight: True
[R2_remove_oof_top100_dup_weight] effective sample size sum: 964.84375
[R2_remove_oof_top100_dup_weight] effective sample weight mean: 0.9571862599206349
R2_remove_oof_top100_dup_weight epoch=01/10 | train_loss=0.5645 train_auc=0.7744 | val_all_auc=0.7800 val_without_top100_auc=0.8336 val_top100_only_auc=0.0545 | 6.9s
  -> saved best checkpoint: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_retrain_oof_top100_20260702_093712/checkpoints/R2_remove_oof_top100_dup_weight/R2_remove_oof_top100_dup_weight_best_val_all_auc.pt
R2_remove_oof_top100_dup_weight epoch=02/10 | train_loss=0.2159 train_auc=0.9744 | val_all_auc=0.7852 val_without_top100_auc=0.8411 val_top100_only_auc=0.0727 | 6.3s
  -> saved best checkpoint: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_retrain_oof_top100_20260702_093712/checkpoints/R2_remove_oof_top100_dup_weight/R2_r

,subset,n,label0,label1,loss,acc,auc,mean_dusty_prob,run,epoch
0,val_all,274,163,111,0.851076,0.740876,0.793732,0.346828,R0,5
1,val_without_top100,258,158,100,0.660200,0.782946,0.842278,0.346472,R0,5
2,val_top100_only,16,5,11,3.928942,0.062500,0.018182,0.352573,R0,5
3,val_all,274,163,111,0.702648,0.751825,0.817941,0.457471,R1_remove_oof_top100,3
4,val_without_top100,258,158,100,0.566583,0.790698,0.868291,0.463530,R1_remove_oof_top100,3
5,val_top100_only,16,5,11,2.896696,0.125000,0.072727,0.359765,R1_remove_oof_top100,3
6,val_all,274,163,111,0.733295,0.748175,0.810756,0.480732,R2_remove_oof_top100_dup_weight,3
7,val_without_top100,258,158,100,0.602032,0.782946,0.861709,0.486311,R2_remove_oof_top100_dup_weight,3
8,val_top100_only,16,5,11,2.849908,0.187500,0.090909,0.390766,R2_remove_oof_top100_dup_weight,3


## 11. R0/R1/R2 Submission Difference Comparison

In [13]:
available_submissions = {}
for run_name, path in submission_paths.items():
    if Path(path).exists():
        df = pd.read_csv(path)
        df["id"] = df["id"].astype(str)
        available_submissions[run_name] = df.rename(columns={"dusty_prob": f"dusty_prob_{run_name}"})

if len(available_submissions) < 2:
    print("Need at least two submissions to compare differences.")
else:
    diff_detail = submission_template[["id"]].copy()
    diff_detail["id"] = diff_detail["id"].astype(str)
    for run_name, df in available_submissions.items():
        diff_detail = diff_detail.merge(df, on="id", how="left")

    if diff_detail.isna().any().any():
        raise ValueError("submission comparison has missing values")

    diff_summary_rows = []
    for left, right in combinations(sorted(available_submissions.keys()), 2):
        left_col = f"dusty_prob_{left}"
        right_col = f"dusty_prob_{right}"
        diff = diff_detail[right_col] - diff_detail[left_col]
        abs_diff = diff.abs()
        diff_summary_rows.append({
            "left": left,
            "right": right,
            "mean_diff_right_minus_left": float(diff.mean()),
            "mean_abs_diff": float(abs_diff.mean()),
            "max_abs_diff": float(abs_diff.max()),
            "q50_abs_diff": float(abs_diff.quantile(0.50)),
            "q90_abs_diff": float(abs_diff.quantile(0.90)),
            "q99_abs_diff": float(abs_diff.quantile(0.99)),
            "pearson_corr": float(diff_detail[[left_col, right_col]].corr(method="pearson").iloc[0, 1]),
            "spearman_corr": float(diff_detail[[left_col, right_col]].corr(method="spearman").iloc[0, 1]),
        })
        diff_detail[f"diff_{right}_minus_{left}"] = diff
        diff_detail[f"abs_diff_{right}_vs_{left}"] = abs_diff

    diff_summary_df = pd.DataFrame(diff_summary_rows)
    diff_detail.to_csv(TABLE_DIR / "submission_pairwise_diff_detail.csv", index=False)
    diff_summary_df.to_csv(TABLE_DIR / "submission_diff_summary.csv", index=False)
    print("saved submission comparison tables")
    display(diff_summary_df)


saved submission comparison tables


,left,right,mean_diff_right_minus_left,mean_abs_diff,max_abs_diff,q50_abs_diff,q90_abs_diff,q99_abs_diff,pearson_corr,spearman_corr
0,R0,R1,0.142766,0.242090,0.996135,0.120954,0.712121,0.952576,0.634427,0.680508
1,R0,R2,0.165802,0.255627,0.996191,0.139978,0.734973,0.963858,0.624097,0.678314
2,R1,R2,0.023036,0.042237,0.394479,0.019409,0.117473,0.269141,0.985819,0.989098


## 12. Final Artifact Checklist

In [14]:
artifact_paths = [
    CONFIG_PATH,
    TABLE_DIR / "oof_validation_summary.csv",
    TABLE_DIR / "oof_checkpoint_audit.csv",
    TABLE_DIR / "oof_vs_v2_compare.csv",
    TABLE_DIR / "topk_baseline_split_label_distribution.csv",
    TABLE_DIR / "removed_mislabel_top100.csv",
    VIS_DIR / "oof_mislabel_top100_grid.png",
    TABLE_DIR / "class_balance_before_after.csv",
    TABLE_DIR / "sample_weight_stats.csv",
    TABLE_DIR / "val_metrics_by_subset.csv",
    TABLE_DIR / "submission_diff_summary.csv",
]

checklist_df = pd.DataFrame({
    "path": [str(path) for path in artifact_paths],
    "exists": [Path(path).exists() for path in artifact_paths],
})
checklist_df.to_csv(TABLE_DIR / "artifact_checklist.csv", index=False)
display(checklist_df)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("Note: If the OOF source was rebuilt from best-AUC checkpoints, this notebook used the rebuilt table under its own output folder.")


,path,exists
0,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True
1,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True
2,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True
3,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True
4,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True
5,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True
6,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True
7,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True
8,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True
9,/home/thkim0/github/AIFFEL_quest_rs/MainQuest/...,True


OUTPUT_DIR: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_retrain_oof_top100_20260702_093712
Note: If the OOF source was rebuilt from best-AUC checkpoints, this notebook used the rebuilt table under its own output folder.
